# 01 · Define & Explore — fibril vs monomer, the epitope, conformational metrics

**Standard slot:** *define & explore.* **For Project 11 this means:** understand amyloid/fibril
structural biology, **choose the target conformation** (the ordered cross-β fibril surface, NOT the
disordered monomer), pick the fibril-surface **epitope residues**, write down the binder + the
**conformational-specificity** metrics, and run a deterministic **mock** mini-run as your
"hello-world" (D0).

Run `00_setup.ipynb` first in this session. A real binder campaign wants an **A100** (see
`MANUAL.md §2`); everything here runs on a no-GPU **mock** backend so you can build the plumbing
anywhere, then switch to the real backend on Colab Pro / A100.

> **The whole project in one sentence:** design a binder that grips the **fibril** conformation of tau
> (Alzheimer's) or α-synuclein (Parkinson's) and **ignores the monomer** — the basis of a
> conformation-selective diagnostic (PET tracer / assay) or an aggregation modulator.

## Why conformational specificity is the hard part

Tau and α-synuclein are **intrinsically disordered as monomers** — no single stable fold. In disease
they stack into **amyloid fibrils**: an ordered, repetitive **cross-β** core with a defined,
solvent-exposed surface (solved by cryo-EM). A useful binder here must do something a normal binder
does not have to do: **discriminate two conformations of the same protein.** It must recognize the
fibril surface and **reject the monomer**, because the monomer is abundant everywhere and a
cross-reactive binder is useless as a fibril-specific tracer.

This is genuinely hard, and **most designs will NOT be selective.** Be honest about it: report the
fraction that pass the fibril filter AND clear the monomer counter-test, not just the best one.

## The metrics, precisely (binder metrics + the specificity gap)

| Metric | Range | Means | Does **not** mean |
|--------|-------|-------|-------------------|
| pLDDT | 0–100 | per-residue *local* confidence of the binder | thermostability / ΔG |
| **pae_interaction** | Å | AF2-Multimer error across the **binder–fibril interface** (key binder metric) | measured affinity |
| scRMSD | Å | designed-vs-predicted Cα-RMSD (self-consistency) | binding/function |
| rosetta_dG | REU | interface energy (more negative = stronger) | a guarantee it binds |
| shape complementarity | 0–1 | interface packing quality | epitope correctness |
| **pae_monomer** | Å | pae_interaction vs the **monomer** conformer (counter-test) | a measured off-rate |
| **specificity_gap** | Å | `pae_monomer − pae_fibril` (positive & large = prefers fibril) | a measured fold-selectivity |

The shared `"binder"` cutoffs: **scRMSD ≤ 2.5, pLDDT ≥ 80, pae_interaction ≤ 10, rosetta_dG ≤ −30,
sc ≥ 0.6.** `pae_interaction` is the single most important binder metric — but a low value is
*confidence*, **not** affinity. The **`specificity_gap` is the project's signature metric** (notebook
04): it is a teaching proxy on a model metric, NOT a measured selectivity, and the monomer is
disordered so its model carries extra uncertainty. A passing, "selective" design is a **hypothesis**
until a **fibril-vs-monomer ELISA/SPR** (notebook 05).

## Setup paths

In [ ]:
import sys, os
# Make the project's scripts/ and the cohort's shared/ importable.
# Adjust these if your Colab working directory differs (see 00_setup §5 for Drive mounting).
sys.path.insert(0, os.path.abspath("../scripts"))
sys.path.insert(0, os.path.abspath("../../../shared"))
os.makedirs("results", exist_ok=True)
print("paths ready; cwd =", os.getcwd())

## 1 · Target conformation + fibril-surface epitope

The design target is **one protofilament surface** of an amyloid fibril — tau paired-helical filament
(PHF) or an α-synuclein fibril — and the hotspots are the **exposed cross-β surface residues** the
binder grips. Steering the binder onto a fibril-specific surface (one that is buried or simply absent
in the disordered monomer) is what makes it *conformation-selective*. Fetch the candidate cryo-EM
fibrils with `data/download_data.py` (tau PHF **5O3L / 5O3T**, α-syn fibril **6CU7 / 6H6B** —
**verify on RCSB**), isolate a protofilament, keep the ordered core, and read the exposed surface
residues off the structure.

Below we just *declare* an EXAMPLE epitope set so the notebook runs end-to-end; **replace it with the
residues you derive from the actual fibril surface** (numbering depends on the PDB you verify).

In [ ]:
import binder_tools as bt

# Two candidate amyloid targets (pick one to design against; cross-amyloid specificity is an extension).
#   TAU_PHF      tau paired-helical filament   (candidate cryo-EM: 5O3L / 5O3T — VERIFY on RCSB)
#   ASYN_FIBRIL  alpha-synuclein fibril        (candidate cryo-EM: 6CU7 / 6H6B — VERIFY on RCSB)
TARGET = "TAU_PHF"                  # one protofilament surface (you extract this from 5O3L/5O3T)

# EXAMPLE fibril-surface hotspots — VERIFY/REPLACE from the cryo-EM fibril surface (data/README.md).
# Real numbering depends on the PDB you clean; tau PHF ordered core is ~ residues 306-378 (R3-R4 repeats).
HOTSPOTS = bt.parse_hotspots("A306,A310,A315,A320")   # EXAMPLE_DATA placeholder fibril-surface residues
print("target  :", TARGET, " (the FIBRIL conformation — not the disordered monomer)")
print("hotspots:", HOTSPOTS, " (EXAMPLE — replace with your verified fibril-surface residues)")
print("\nCounter-test conformer: the disordered MONOMER (an AFDB/ensemble model — a modeling caveat).")

## 2 · Mock hello-world: a tiny two-paradigm mini-run

`scripts/binder_tools.py` exposes both paradigms behind one API:
`generate_binders_bindcraft(...)` and `generate_binders_rfdiffusion(...)`, plus `af2_multimer(...)`
(the FIBRIL-state scorer) and the project's HARD-PART helpers `conformational_specificity(...)` /
`specificity_gap(...)`. The **mock** backend is deterministic and GPU-free so you can develop the
plumbing. **Never report mock numbers as real** — they are `SYNTHETIC` by construction.

In [ ]:
# A few designs from each paradigm, scored by mock AF2-Multimer (FIBRIL state). Numbers are SYNTHETIC.
bc = bt.generate_binders_bindcraft(TARGET, HOTSPOTS, n=3, tool="mock")
rf = bt.generate_binders_rfdiffusion(TARGET, HOTSPOTS, n=3, tool="mock")
bt.score_designs(bc, tool="mock")
bt.score_designs(rf, tool="mock")

d = bc[0]
print("example BindCraft design:")
print("  id   :", d.design_id)
print("  len  :", d.length, "aa")
print("  seq  :", d.sequence)
print("  pae_interaction (fibril) =", d.pae_interaction, " scrmsd =", d.scrmsd,
      " sc =", d.shape_complementarity, " (SYNTHETIC)")
print("  synthetic flag  :", d.synthetic, "->", d.notes[0])
print("\nReminder: switch tool='mock' -> 'bindcraft'/'rfdiffusion'/'af2' on Colab (A100). See MANUAL.md §2.")

## 3 · The HARD PART, previewed: fibril vs monomer

A fibril binder is only useful if it **prefers the fibril over the monomer**. `conformational_specificity()`
scores the binder against each conformer; `specificity_gap()` = `pae_monomer − pae_fibril` (positive &
large ⇒ prefers the fibril). This is the heart of notebook 04. Mock numbers are SYNTHETIC and the
monomer/fibril bias here is a **teaching device**, not a claim of real selectivity.

In [ ]:
bt.evaluate_specificity(bc, tool="mock")   # scores BOTH conformers, fills pae_fibril/pae_monomer/specificity_gap
bt.evaluate_specificity(rf, tool="mock")

for b in bc[:3]:
    sel = "fibril-selective?" if (b.specificity_gap or 0) > 0 else "NOT selective"
    print(f"{b.design_id}: pae_fibril={b.pae_fibril}  pae_monomer={b.pae_monomer}  "
          f"gap={b.specificity_gap}  -> {sel}  (SYNTHETIC)")
print("\nA positive gap means AF2 is MORE confident about the fibril complex than the monomer one.")
print("This is a HYPOTHESIS — only a fibril-vs-monomer ELISA/SPR (nb 05) proves selectivity.")

## 4 · Epitope-coverage proxy (does it cover the fibril surface?)

A binder can only be a fibril tracer if it actually sits on the exposed fibril epitope.
`hotspot_overlap()` is a geometry proxy (fraction of fibril-surface hotspots contacted) — a teaching
stand-in for the structural epitope mapping you would confirm experimentally. Higher ⇒ more of the
fibril surface covered (but coverage alone does NOT imply monomer rejection — that is the gap test).

In [ ]:
for b in bc[:3]:
    ov = bt.hotspot_overlap(b.contact_residues, HOTSPOTS)
    print(f"{b.design_id}: contacts {b.contact_residues} -> fibril-epitope coverage = {ov} (SYNTHETIC)")

## Visualize a binder–fibril complex (py3Dmol)

Use this to eyeball a predicted binder–fibril complex once you have a real PDB (from AF2-Multimer), or
just to inspect the cryo-EM fibril surface you are targeting.

In [ ]:
import py3Dmol

def show_complex(pdb_path_or_str, is_path=True):
    data = open(pdb_path_or_str).read() if is_path else pdb_path_or_str
    view = py3Dmol.view(width=520, height=420)
    view.addModel(data, "pdb")
    view.setStyle({"cartoon": {"color": "spectrum"}})
    view.zoomTo()
    return view.show()

# Example (after a real AF2-Multimer prediction writes a complex PDB, or to view the fibril):
# show_complex("results/af2/top_fibril_complex.pdb")
# show_complex("data/inputs/5O3L_protofilament.pdb")
print("show_complex(pdb_path) ready.")

## D0 checklist
- [ ] Fibril accessions verified on RCSB (tau **5O3L/5O3T**, α-syn **6CU7/6H6B** are candidates); protofilament + ordered core identified.
- [ ] **Target conformation chosen** (the fibril surface) + a fibril-surface **epitope list** (derived from the structure, not invented).
- [ ] A monomer model assembled for the counter-test (AFDB/ensemble — note the disorder caveat).
- [ ] One-paragraph definition of each metric **with** its "does not mean" note, including `specificity_gap`.
- [ ] Reproduced mock mini-run (both paradigms) with metrics + the monomer-vs-fibril gap printed and flagged SYNTHETIC.
- [ ] Problem statement with measurable success criteria + controls (fibril-vs-monomer, scrambled-interface); `LOG.md` entry (GPU, seed).

**Next:** `02_generate.ipynb` — the two-paradigm binder campaign to the fibril epitope.